<a href="https://colab.research.google.com/github/rm-cg/commercial-energy-optimization-synthetic-data/blob/main/Deliverable_2_Python_Generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Set a random seed for reproducibility
np.random.seed(42)

# --- 1. Generate Static Table: buildings ---
num_buildings = 100
buildings = pd.DataFrame({
    'building_id': range(1, num_buildings + 1),
    'primary_activity': np.random.choice([
        # Offices & Workspaces
        'Corporate Office', 'Co-working Space',
        # Retail & Groceries
        'Stand-Alone Retail Store', 'Strip Mall', 'Supermarket',
        # Food & Beverage
        'Fast Food Restaurant', 'Fine Dining Restaurant', 'Coffee Shop',
        # Education
        'Primary School', 'University Building',
        # Healthcare (Clinics & Hospitals)
        'General Hospital', 'Outpatient Clinic', 'Dental Clinic', 'Nursing Home',
        # Residential & Lodging
        'Residential Apartment Building', 'Townhouse Complex', 'Boutique Hotel',
        # Industrial & Leisure
        'Non-Refrigerated Warehouse', 'Indoor Sports Center', 'Movie Theater'
    ], size=num_buildings),
    'floor_area_sqft': np.round(np.random.uniform(5000, 50000, size=num_buildings), 2)
})
# Assign Base HVAC Capacity based on floor area
buildings['base_hvac_capacity'] = np.round(buildings['floor_area_sqft'] / 1000 * 1.5, 2)

# --- 2. Generate Static Table: meters ---
# Applying the 5% Real-World Messiness rule for broken sensors
meters = pd.DataFrame({
    'meter_id': range(1, num_buildings + 1),
    'building_id': buildings['building_id'],
    'utility_type': 'electricity',
    'is_broken': np.random.choice([0, 1], size=num_buildings, p=[0.95, 0.05])
})

# --- 3. Generate Static Table: tariffs ---
# Applying the proven Inclining Block Rates from economics model
tariffs = pd.DataFrame({
    'tariff_id': [2, 3],
    'time_of_use_tier': ['Off-Peak (Base)', 'Peak (Stress)'],
    'rate_per_kwh': [0.10, 0.30]
})

# Display the first few rows to verify our data types match the Data Dictionary
print("--- BUILDINGS TABLE ---")
print(buildings.head(20))
print("\n--- METERS TABLE ---")
print(meters.head(20))
print("\n--- TARIFFS TABLE ---")
print(tariffs)

--- BUILDINGS TABLE ---
    building_id                primary_activity  floor_area_sqft  \
0             1          Fine Dining Restaurant         37832.28   
1             2                   Movie Theater         33690.09   
2             3  Residential Apartment Building         44924.57   
3             4                General Hospital         26249.67   
4             5                     Coffee Shop         10381.74   
5             6          Fine Dining Restaurant         37096.02   
6             7            Indoor Sports Center         39235.33   
7             8                General Hospital         30257.47   
8             9                General Hospital         39693.52   
9            10                      Strip Mall         27220.80   
10           11                     Coffee Shop         28522.98   
11           12        Stand-Alone Retail Store         24239.35   
12           13                Co-working Space          6143.86   
13           14         

In [ ]:
# 1. Time-Series Timeline (1 month of hourly data)
# 30 days * 24 hours = 720 hours of data per building
date_range = pd.date_range(start='2026-06-01', periods=720, freq='h')

# 2. Ambient Weather Forcing (The Sine Wave Engine)
# T_ambient = T_mean + T_amp * sin(pi * (t - 8) / 12)
hour_of_day = date_range.hour
t_mean = 30.0  # Base temperature of 30°C
t_amp = 5.0    # Swings up to 35°C in the afternoon
ambient_temp_c = t_mean + t_amp * np.sin(np.pi * (hour_of_day - 8) / 12)

# 3. Thermal Velocity (The Calculus Component: dT/dt)

temp_diff = np.diff(ambient_temp_c, prepend=ambient_temp_c[0])
v_thermal = np.maximum(0, temp_diff) # only trigger overdrive when temp is RISING

# Constants for the Physics Model
alpha = 1.5 # Static thermal leakage coefficient
beta = 0.5  # Kinetic resistance coefficient

occupancy_list = []
meter_readings_list = []
reading_id_counter = 1
occupancy_id_counter = 1

# 4. Generate the loads building-by-building
for idx, building in buildings.iterrows():
    b_id = building['building_id']
    base_hvac = building['base_hvac_capacity']

    # Probabilistic Human Load (Poisson Distribution)
    # Estimate building capacity based on floor area (e.g., 1 person per 200 sqft)
    max_capacity = building['floor_area_sqft'] / 200
    # Schedule (lambda): High occupancy 9 AM to 5 PM, low otherwise
    lambda_t = np.where((hour_of_day >= 9) & (hour_of_day <= 17), max_capacity * 0.8, max_capacity * 0.05)
    headcount = np.random.poisson(lam=lambda_t)

    # Build the 'occupancy' table for this building
    occ_df = pd.DataFrame({
        'occupancy_id': range(occupancy_id_counter, occupancy_id_counter + len(date_range)),
        'building_id': b_id,
        'timestamp': date_range,
        'headcount': headcount
    })
    occupancy_list.append(occ_df)
    occupancy_id_counter += len(date_range)

    # Formulate the Custom Energy Equations
    # A. Human Heat (100 Watts per person)
    q_human = (headcount * 100) / 1000

    # B. Kinetic Overdrive
    e_overdrive = beta * (v_thermal ** 2)

    # C. Total Energy Consumed (Base)
    # apply the hidden faulty insulation penalty in Day 3!
    energy_consumed = base_hvac + q_human + (alpha * (ambient_temp_c - 25)) + e_overdrive

    # matching meter_id for this building
    m_id = meters[meters['building_id'] == b_id]['meter_id'].values[0]

    # the 'meter_readings' table
    readings_df = pd.DataFrame({
        'reading_id': range(reading_id_counter, reading_id_counter + len(date_range)),
        'meter_id': m_id,
        'timestamp': date_range,
        'ambient_temp_c': np.round(ambient_temp_c, 2),
        'energy_consumed': np.round(energy_consumed, 2)
    })
    meter_readings_list.append(readings_df)
    reading_id_counter += len(date_range)

# Combine all lists into our final DataFrames
occupancy = pd.concat(occupancy_list, ignore_index=True)
meter_readings = pd.concat(meter_readings_list, ignore_index=True)

# Display the data to verify
print(f"--- OCCUPANCY TABLE (Total Rows: {len(occupancy)}) ---")
print(occupancy.head(20))
print(f"\n--- METER READINGS TABLE (Total Rows: {len(meter_readings)}) ---")
print(meter_readings.head(20))


--- OCCUPANCY TABLE (Total Rows: 72000) ---
    occupancy_id  building_id           timestamp  headcount
0              1            1 2026-06-01 00:00:00          5
1              2            1 2026-06-01 01:00:00          8
2              3            1 2026-06-01 02:00:00         11
3              4            1 2026-06-01 03:00:00         13
4              5            1 2026-06-01 04:00:00          6
5              6            1 2026-06-01 05:00:00          5
6              7            1 2026-06-01 06:00:00          8
7              8            1 2026-06-01 07:00:00          7
8              9            1 2026-06-01 08:00:00         13
9             10            1 2026-06-01 09:00:00        158
10            11            1 2026-06-01 10:00:00        145
11            12            1 2026-06-01 11:00:00        143
12            13            1 2026-06-01 12:00:00        159
13            14            1 2026-06-01 13:00:00        168
14            15            1 2026-06-01 

In [ ]:
import numpy as np
import pandas as pd

num_buildings = 100

# 1. Inject the "Faulty Insulation" Omitted Variable
# Randomly select 20% of our buildings to suffer from faulty insulation (multiplier between 1.15 and 1.30)
buildings['faulty_insulation_multiplier'] = np.where(
    np.random.rand(num_buildings) < 0.20,
    np.random.uniform(1.15, 1.30, size=num_buildings),
    1.0
)

# Temporarily map the multipliers to our meter_readings table to apply the math
meter_to_building = meters.set_index('meter_id')['building_id'].to_dict()
building_to_multiplier = buildings.set_index('building_id')['faulty_insulation_multiplier'].to_dict()

meter_readings['building_id'] = meter_readings['meter_id'].map(meter_to_building)
meter_readings['hidden_multiplier'] = meter_readings['building_id'].map(building_to_multiplier)

# Multiply the baseline energy consumed by the hidden penalty
meter_readings['energy_consumed'] = meter_readings['energy_consumed'] * meter_readings['hidden_multiplier']

# 1.5 Inject the "Unreported Overtime" Omitted Variable
# Randomly select 20% of buildings to have unlogged employees staying late
buildings['unreported_overtime_multiplier'] = np.where(
    np.random.rand(num_buildings) < 0.20,
    1.20,
    1.0
)

# Temporarily map the overtime multiplier to the meter_readings table
building_to_overtime = buildings.set_index('building_id')['unreported_overtime_multiplier'].to_dict()
meter_readings['overtime_multiplier'] = meter_readings['building_id'].map(building_to_overtime)

# Ensure the timestamp is a datetime object so we can extract the hour
meter_readings['timestamp'] = pd.to_datetime(meter_readings['timestamp'])

# Apply the 20% energy penalty ONLY between 18:00 and 21:00
mask_evening = meter_readings['timestamp'].dt.hour.isin([18, 19, 20, 21])
meter_readings.loc[mask_evening, 'energy_consumed'] = meter_readings.loc[mask_evening, 'energy_consumed'] * meter_readings.loc[mask_evening, 'overtime_multiplier']

# Drop the temporary multiplier from meter_readings
meter_readings = meter_readings.drop(columns=['overtime_multiplier'])

# 2. Add Real-World Measurement Noise (Gaussian Noise +/- 2%)
# use a normal distribution centered at 1.0 with a standard deviation of 0.01
noise = np.random.normal(loc=1.0, scale=0.01, size=len(meter_readings))
meter_readings['energy_consumed'] = np.round(meter_readings['energy_consumed'] * noise, 2)

# 3. Simulate Broken Sensors (Missing Data / NaNs)
# Find the meters where the 'is_broken' flag is exactly 1
broken_meters = meters[meters['is_broken'] == 1]['meter_id'].tolist()

# Create a mask to randomly select 5% of the readings strictly from those broken meters
mask_broken = meter_readings['meter_id'].isin(broken_meters)
mask_nan = mask_broken & (np.random.rand(len(meter_readings)) < 0.05)

# Replace the energy_consumed values with NaN (blank) for the selected rows
meter_readings.loc[mask_nan, 'energy_consumed'] = np.nan

# 4. The Golden Rule of Omitted Variables: Delete the Evidence!

meter_readings = meter_readings.drop(columns=['building_id', 'hidden_multiplier'])
buildings = buildings.drop(columns=['faulty_insulation_multiplier'])

# Display a sample of readings from a broken meter to verify our NaNs and noise!
if len(broken_meters) > 0:
    print(f"--- SAMPLE READINGS FROM BROKEN METER ID: {broken_meters} ---")
    print(meter_readings[meter_readings['meter_id'].isin(broken_meters)].head(15))
else:
    print("Wow, a rare 5% probability event: None of your 100 meters broke!")

--- SAMPLE READINGS FROM BROKEN METER ID: [41, 56, 62, 63, 76] ---
       reading_id  meter_id           timestamp  ambient_temp_c  \
28800       28801        41 2026-06-01 00:00:00           25.67   
28801       28802        41 2026-06-01 01:00:00           25.17   
28802       28803        41 2026-06-01 02:00:00           25.00   
28803       28804        41 2026-06-01 03:00:00           25.17   
28804       28805        41 2026-06-01 04:00:00           25.67   
28805       28806        41 2026-06-01 05:00:00           26.46   
28806       28807        41 2026-06-01 06:00:00           27.50   
28807       28808        41 2026-06-01 07:00:00           28.71   
28808       28809        41 2026-06-01 08:00:00           30.00   
28809       28810        41 2026-06-01 09:00:00           31.29   
28810       28811        41 2026-06-01 10:00:00           32.50   
28811       28812        41 2026-06-01 11:00:00           33.54   
28812       28813        41 2026-06-01 12:00:00           34.3

In [ ]:
from google.colab import files
import numpy as np

# Expanded arrays: 20 streets, and 20 places per region (Luzon, Visayas, Mindanao)
streets = [
    'Rizal Ave', 'Ayala Ave', 'Quezon Blvd', 'Taft Ave', 'Bonifacio St',
    'Osmena Blvd', 'Colon St', 'Jones Ave', 'Magallanes St', 'San Pedro St',
    'Claro M. Recto Ave', 'Session Road', 'McArthur Highway', 'Lacson St',
    'Roxas Blvd', 'Espana Blvd', 'Magsaysay Ave', 'Burgos St', 'Luna St', 'Del Pilar St'
]

cities = [
    # Luzon (20 places)
    'Manila', 'Makati', 'Quezon City', 'Baguio', 'Legazpi', 'Laoag',
    'Cavite', 'Laguna', 'Batangas', 'Rizal', 'Bulacan', 'Pampanga',
    'Tarlac', 'Zambales', 'Bataan', 'Pangasinan', 'La Union', 'Ilocos Sur',
    'Isabela', 'Cagayan',

    # Visayas (20 places)
    'Cebu City', 'Iloilo City', 'Bacolod', 'Dumaguete', 'Tacloban',
    'Leyte', 'Bohol', 'Capiz', 'Negros Occidental', 'Antique',
    'Aklan', 'Eastern Samar', 'Negros Oriental', 'Western Samar', 'Northern Samar',
    'Southern Leyte', 'Biliran', 'Guimaras', 'Siquijor', 'Roxas City',

    # Mindanao (20 places)
    'Davao City', 'Cagayan de Oro', 'Zamboanga City', 'General Santos', 'Iligan',
    'Bukidnon', 'Agusan del Sur', 'Davao del Sur', 'Zamboanga del Sur', 'Maguindanao',
    'Davao Oriental', 'Zamboanga del Norte', 'Sultan Kudarat', 'Lanao del Sur', 'South Cotabato',
    'Surigao del Sur', 'North Cotabato', 'Sarangani', 'Misamis Oriental', 'Lanao del Norte'
]

# Generate a random address for each building
buildings['address'] = [f"{np.random.randint(1, 9999)} {np.random.choice(streets)}, {np.random.choice(cities)}" for _ in range(num_buildings)]

print("--- Final Buildings Table (With Text Generation) ---")
print(buildings.head())

# ---> THE OMITTED VARIABLE DROP GOES HERE <---
# Delete BOTH secret multipliers from the buildings table safely
buildings = buildings.drop(columns=['faulty_insulation_multiplier', 'unreported_overtime_multiplier'], errors='ignore')

# Export all 5 tables to CSV files
buildings.to_csv('synthetic_buildings.csv', index=False)
meters.to_csv('synthetic_meters.csv', index=False)
tariffs.to_csv('synthetic_tariffs.csv', index=False)
occupancy.to_csv('synthetic_occupancy.csv', index=False)
meter_readings.to_csv('synthetic_meter_readings.csv', index=False)


--- Final Buildings Table (With Text Generation) ---
   building_id                primary_activity  floor_area_sqft  \
0            1          Fine Dining Restaurant         37832.28   
1            2                   Movie Theater         33690.09   
2            3  Residential Apartment Building         44924.57   
3            4                General Hospital         26249.67   
4            5                     Coffee Shop         10381.74   

   base_hvac_capacity  unreported_overtime_multiplier  \
0               56.75                             1.2   
1               50.54                             1.2   
2               67.39                             1.2   
3               39.37                             1.0   
4               15.57                             1.0   

                               address  
0               3290 Taft Ave, Biliran  
1               3915 Jones Ave, Laguna  
2            9295 Burgos St, Dumaguete  
3  6056 Claro M. Recto Ave, Roxas Cit